## 1. Setup Mountainsort
Before this notebook, need to do 
- (1) Clusterless sorters Notebook 5 in the General Dataprocessing first. That notebook detects lick artifact time.
- (2) decodePrep in Notebook 6. This step unions artifact times detected from the left and right hesmisphere and propagate to every tetrode so that every tetrode has the same artifact times. In addition, this step intersects with the track time obtained from Statescript.)

In [108]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [41]:
import os
#import cupy as cp
import numpy as np
import datajoint as dj
import spyglass as nd
import pandas as pd
import matplotlib.pyplot as plt
import json
import multiprocessing

# ignore datajoint+jupyter async warnings
import warnings
warnings.simplefilter('ignore', category=DeprecationWarning)
warnings.simplefilter('ignore', category=ResourceWarning)

from spyglass.common import (Session, IntervalList,LabMember, LabTeam, Raw, Session, Nwbfile,
                            Electrode,LFPBand,interval_list_intersect, RawPosition, TaskEpoch)
import spyglass.spikesorting as ss

In [42]:
from spyglass.spikesorting.v0 import (SortGroup, 
                                    SortInterval,
                                    SpikeSortingPreprocessingParameters,
                                    SpikeSortingRecording, 
                                    SpikeSorterParameters,
                                    SpikeSortingRecordingSelection,
                                    ArtifactDetectionParameters, ArtifactDetectionSelection,
                                    ArtifactRemovedIntervalList, ArtifactDetection,
                                      SpikeSortingSelection, SpikeSorting,
                                   CuratedSpikeSortingSelection,CuratedSpikeSorting,Curation)
from spyglass.spikesorting.v0.curation_figurl import CurationFigurl,CurationFigurlSelection
from spyglass.spikesorting.v0.spikesorting_curation import MetricParameters,MetricSelection,QualityMetrics
from spyglass.spikesorting.v0.spikesorting_curation import WaveformParameters,WaveformSelection,Waveforms
from spyglass.utils.nwb_helper_fn import get_nwb_copy_filename
from pprint import pprint

from spyglass.shijiegu.helpers import interval_union
from spyglass.shijiegu.Analysis_SGU import TrialChoice,RippleTimes,EpochPos
from spyglass.shijiegu.load import load_run_sessions
from spyglass.shijiegu.singleUnit import do_mountainSort
from spyglass.shijiegu.decodeHelpers import runSessionNames,sleepSessionNames

### 0. Double check parameter

In [43]:
sorter_params_name = "CA1_tet_Shijie_whiten"
sorter_params = (SpikeSorterParameters & {"sorter_params_name": sorter_params_name}).fetch1("sorter_params")
sorter_params

{'detect_sign': -1,
 'adjacency_radius': -1,
 'freq_min': 0,
 'freq_max': 0,
 'filter': False,
 'whiten': True,
 'num_workers': 4,
 'clip_size': 39,
 'detect_threshold': 3,
 'detect_interval': 10,
 'verbose': True}

### 1. specify data

In [ ]:
nwb_copy_file_name = "julio20230807_.nwb"

In [ ]:
SortInterval & {'nwb_file_name': nwb_copy_file_name}

In [ ]:
intervals, _ = runSessionNames(nwb_copy_file_name)
print(intervals)

In [ ]:
intervals2, _ = sleepSessionNames(nwb_copy_file_name)
print(intervals2)

### 2. Do sorting

In [ ]:
tetrode_with_cell=np.unique((SpikeSortingRecordingSelection & {'nwb_file_name':nwb_copy_file_name}).fetch('sort_group_id'))

In [ ]:
tetrode_with_cell

In [ ]:
for session_name in intervals:
    do_mountainSort(nwb_copy_file_name,session_name)

In [ ]:
for session_name in intervals2:
    do_mountainSort(nwb_copy_file_name,session_name)

[10:59:55][INFO] Spyglass: Running spike sorting on {'nwb_file_name': 'eliot20221023_.nwb', 'sort_group_id': 0, 'sort_interval_name': '01_Seq2Sleep1', 'preproc_params_name': 'franklab_tetrode_hippocampus', 'team_name': 'Shijie Gu', 'sorter': 'mountainsort4', 'sorter_params_name': 'CA1_tet_Shijie_whiten', 'artifact_removed_interval_list_name': 'eliot20221023_.nwb_01_Seq2Sleep1_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only'}...
INFO:spyglass:Running spike sorting on {'nwb_file_name': 'eliot20221023_.nwb', 'sort_group_id': 0, 'sort_interval_name': '01_Seq2Sleep1', 'preproc_params_name': 'franklab_tetrode_hippocampus', 'team_name': 'Shijie Gu', 'sorter': 'mountainsort4', 'sorter_params_name': 'CA1_tet_Shijie_whiten', 'artifact_removed_interval_list_name': 'eliot20221023_.nwb_01_Seq2Sleep1_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only'}...


Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording
Using temporary directory /stelmo/nwb/tmp/tmp674uomux
Using 4 workers.
Using tempdir: /stelmo/nwb/tmp/tmp674uomux/tmpipksncmk
Num. workers = 4
Preparing /stelmo/nwb/tmp/tmp674uomux/tmpipksncmk/timeseries.hdf5...
Preparing neighborhood sorters (M=4, N=52414690)...
Neighboorhood of channel 3 has 4 channels.
Detecting events on channel 4 (phase1)...
Elapsed time for detect on neighborhood: 0:00:03.502248
Num events detected on channel 4 (phase1): 82134
Computing PCA features for channel 4 (phase1)...
Clustering for channel 4 (phase1)...
Found 4 clusters for channel 4 (phase1)...
Computing templates for channel 4 (phase1)...
Re-assigning events for channel 4 (phase1)...
Re-assigning 4 events from 4 to 2 with dt=-1 (k=2)
Neighboorhood of channel 0 has 4 channels.
Detecting events on channel 1 (phase1)...
Elapsed time for detect on neighborhood: 0:00:03.645195
Num events detected on channel 1 (phase1): 87202
Computing PC

[11:01:21][INFO] Spyglass: Saving sorting results...
INFO:spyglass:Saving sorting results...
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/core/basesorting.py:237: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")
[11:01:23][INFO] Spyglass: Running spike sorting on {'nwb_file_name': 'eliot20221023_.nwb', 'sort_group_id': 2, 'sort_interval_name': '01_Seq2Sleep1', 'preproc_params_name': 'franklab_tetrode_hippocampus', 'team_name': 'Shijie Gu', 'sorter': 'mountainsort4', 'sorter_params_name': 'CA1_tet_Shijie_whiten', 'artifact_removed_interval_list_name': 'eliot20221023_.nwb_01_Seq2Sleep1_2_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only'}...
INFO:spyglass:Running spike sorting on {'nwb_file_name': 'eliot20221023_.nwb', 'sort_group_id': 2, 'sort_interva

Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording
Using temporary directory /stelmo/nwb/tmp/tmps_9lugxb
Using 4 workers.
Using tempdir: /stelmo/nwb/tmp/tmps_9lugxb/tmp7j1u4mmj
Num. workers = 4
Preparing /stelmo/nwb/tmp/tmps_9lugxb/tmp7j1u4mmj/timeseries.hdf5...
Preparing neighborhood sorters (M=4, N=52414690)...
Neighboorhood of channel 1 has 4 channels.
Detecting events on channel 2 (phase1)...
Elapsed time for detect on neighborhood: 0:00:03.469903
Num events detected on channel 2 (phase1): 82062
Computing PCA features for channel 2 (phase1)...
Clustering for channel 2 (phase1)...
Found 3 clusters for channel 2 (phase1)...
Computing templates for channel 2 (phase1)...
Re-assigning events for channel 2 (phase1)...
Re-assigning 43 events from 2 to 3 with dt=4 (k=3)
Neighboorhood of channel 3 has 4 channels.
Detecting events on channel 4 (phase1)...
Elapsed time for detect on neighborhood: 0:00:03.503175
Num events detected on channel 4 (phase1): 76642
Computing PC

[11:02:28][INFO] Spyglass: Saving sorting results...
INFO:spyglass:Saving sorting results...
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/core/basesorting.py:237: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")
[11:02:30][INFO] Spyglass: Running spike sorting on {'nwb_file_name': 'eliot20221023_.nwb', 'sort_group_id': 4, 'sort_interval_name': '01_Seq2Sleep1', 'preproc_params_name': 'franklab_tetrode_hippocampus', 'team_name': 'Shijie Gu', 'sorter': 'mountainsort4', 'sorter_params_name': 'CA1_tet_Shijie_whiten', 'artifact_removed_interval_list_name': 'eliot20221023_.nwb_01_Seq2Sleep1_4_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only'}...
INFO:spyglass:Running spike sorting on {'nwb_file_name': 'eliot20221023_.nwb', 'sort_group_id': 4, 'sort_interva

...

Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording
Using temporary directory /stelmo/nwb/tmp/tmptg674v9k
Using 4 workers.
Using tempdir: /stelmo/nwb/tmp/tmptg674v9k/tmpisuaqmai
Num. workers = 4
Preparing /stelmo/nwb/tmp/tmptg674v9k/tmpisuaqmai/timeseries.hdf5...
Preparing neighborhood sorters (M=4, N=52414690)...
Neighboorhood of channel 1 has 4 channels.
Detecting events on channel 2 (phase1)...
Elapsed time for detect on neighborhood: 0:00:03.820549
Num events detected on channel 2 (phase1): 79660
Computing PCA features for channel 2 (phase1)...
Clustering for channel 2 (phase1)...
Found 2 clusters for channel 2 (phase1)...
Computing templates for channel 2 (phase1)...
Re-assigning events for channel 2 (phase1)...
Neighboorhood of channel 2 has 4 channels.
Detecting events on channel 3 (phase1)...
Elapsed time for detect on neighborhood: 0:00:03.911449
Num events detected on channel 3 (phase1): 95283
Computing PCA features for channel 3 (phase1)...
Clustering for

[11:12:38][INFO] Spyglass: Saving sorting results...
INFO:spyglass:Saving sorting results...
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/core/basesorting.py:237: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")
[11:12:41][INFO] Spyglass: Running spike sorting on {'nwb_file_name': 'eliot20221023_.nwb', 'sort_group_id': 27, 'sort_interval_name': '01_Seq2Sleep1', 'preproc_params_name': 'franklab_tetrode_hippocampus', 'team_name': 'Shijie Gu', 'sorter': 'mountainsort4', 'sorter_params_name': 'CA1_tet_Shijie_whiten', 'artifact_removed_interval_list_name': 'eliot20221023_.nwb_01_Seq2Sleep1_27_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only'}...
INFO:spyglass:Running spike sorting on {'nwb_file_name': 'eliot20221023_.nwb', 'sort_group_id': 27, 'sort_inte

Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording
Using temporary directory /stelmo/nwb/tmp/tmp5u7uqti4
Using 4 workers.
Using tempdir: /stelmo/nwb/tmp/tmp5u7uqti4/tmpm9mg78li
Num. workers = 4
Preparing /stelmo/nwb/tmp/tmp5u7uqti4/tmpm9mg78li/timeseries.hdf5...
Preparing neighborhood sorters (M=4, N=52414690)...
Neighboorhood of channel 1 has 4 channels.
Detecting events on channel 2 (phase1)...
Elapsed time for detect on neighborhood: 0:00:03.867360
Num events detected on channel 2 (phase1): 89657
Computing PCA features for channel 2 (phase1)...
Clustering for channel 2 (phase1)...
Found 3 clusters for channel 2 (phase1)...
Computing templates for channel 2 (phase1)...
Re-assigning events for channel 2 (phase1)...
Re-assigning 651 events from 2 to 4 with dt=-1 (k=2)
Neighboorhood of channel 2 has 4 channels.
Detecting events on channel 3 (phase1)...
Elapsed time for detect on neighborhood: 0:00:03.495351
Num events detected on channel 3 (phase1): 70022
Computing 

[11:13:45][INFO] Spyglass: Saving sorting results...
INFO:spyglass:Saving sorting results...
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/core/basesorting.py:237: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")
[11:13:47][INFO] Spyglass: Running spike sorting on {'nwb_file_name': 'eliot20221023_.nwb', 'sort_group_id': 28, 'sort_interval_name': '01_Seq2Sleep1', 'preproc_params_name': 'franklab_tetrode_hippocampus', 'team_name': 'Shijie Gu', 'sorter': 'mountainsort4', 'sorter_params_name': 'CA1_tet_Shijie_whiten', 'artifact_removed_interval_list_name': 'eliot20221023_.nwb_01_Seq2Sleep1_28_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only'}...
INFO:spyglass:Running spike sorting on {'nwb_file_name': 'eliot20221023_.nwb', 'sort_group_id': 28, 'sort_inte

Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording
Using temporary directory /stelmo/nwb/tmp/tmpvh8zfe1w
Using 4 workers.
Using tempdir: /stelmo/nwb/tmp/tmpvh8zfe1w/tmp7uia7qcn
Num. workers = 4
Preparing /stelmo/nwb/tmp/tmpvh8zfe1w/tmp7uia7qcn/timeseries.hdf5...
Preparing neighborhood sorters (M=4, N=52414690)...
Neighboorhood of channel 3 has 4 channels.
Detecting events on channel 4 (phase1)...
Elapsed time for detect on neighborhood: 0:00:03.831969
Num events detected on channel 4 (phase1): 58459
Computing PCA features for channel 4 (phase1)...
Clustering for channel 4 (phase1)...
Found 2 clusters for channel 4 (phase1)...
Computing templates for channel 4 (phase1)...
Re-assigning events for channel 4 (phase1)...
Neighboorhood of channel 0 has 4 channels.
Detecting events on channel 1 (phase1)...
Elapsed time for detect on neighborhood: 0:00:03.601779
Num events detected on channel 1 (phase1): 57858
Computing PCA features for channel 1 (phase1)...
Clustering for

In [158]:
all_tet_list = np.unique((SpikeSorting & {"nwb_file_name": nwb_copy_file_name,"sorter":"mountainsort4"}).fetch("sort_group_id"))

In [159]:
for tet in all_tet_list:
    key = {'nwb_file_name' : nwb_copy_file_name,
           "sort_group_id": tet,"sorter":"mountainsort4"}
    assert len(SpikeSortingRecordingSelection & key) == len(SpikeSortingRecording & key)
    
(SpikeSorting & {"nwb_file_name": nwb_copy_file_name,"sorter":"mountainsort4"})

nwb_file_name name of the NWB file,sort_group_id identifier for a group of electrodes,sort_interval_name name for this interval,preproc_params_name,team_name,sorter,sorter_params_name,artifact_removed_interval_list_name,sorting_path,"time_of_sort in Unix time, to the nearest second"
eliot20221024_.nwb,0,01_Seq2Sleep1,franklab_tetrode_hippocampus,Shijie Gu,mountainsort4,CA1_tet_Shijie_whiten,eliot20221024_.nwb_01_Seq2Sleep1_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/eliot20221024_.nwb_01_Seq2Sleep1_0_franklab_tetrode_hippocampus_6c9f13d6_spikesorting,1754447297
eliot20221024_.nwb,0,02_Seq2Session2,franklab_tetrode_hippocampus,Shijie Gu,mountainsort4,CA1_tet_Shijie_whiten,eliot20221024_.nwb_02_Seq2Session2_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/eliot20221024_.nwb_02_Seq2Session2_0_franklab_tetrode_hippocampus_02cffd30_spikesorting,1745080472
eliot20221024_.nwb,0,03_Seq2Sleep2,franklab_tetrode_hippocampus,Shijie Gu,mountainsort4,CA1_tet_Shijie_whiten,eliot20221024_.nwb_03_Seq2Sleep2_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/eliot20221024_.nwb_03_Seq2Sleep2_0_franklab_tetrode_hippocampus_24aba637_spikesorting,1754448535
eliot20221024_.nwb,0,04_Seq2Session3,franklab_tetrode_hippocampus,Shijie Gu,mountainsort4,CA1_tet_Shijie_whiten,eliot20221024_.nwb_04_Seq2Session3_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/eliot20221024_.nwb_04_Seq2Session3_0_franklab_tetrode_hippocampus_783e46fb_spikesorting,1745081884
eliot20221024_.nwb,0,05_Seq2Sleep3,franklab_tetrode_hippocampus,Shijie Gu,mountainsort4,CA1_tet_Shijie_whiten,eliot20221024_.nwb_05_Seq2Sleep3_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/eliot20221024_.nwb_05_Seq2Sleep3_0_franklab_tetrode_hippocampus_64eaf11c_spikesorting,1754449872
eliot20221024_.nwb,0,06_Seq2Session4,franklab_tetrode_hippocampus,Shijie Gu,mountainsort4,CA1_tet_Shijie_whiten,eliot20221024_.nwb_06_Seq2Session4_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/eliot20221024_.nwb_06_Seq2Session4_0_franklab_tetrode_hippocampus_de6ab192_spikesorting,1745083411
eliot20221024_.nwb,0,07_Seq2Sleep4,franklab_tetrode_hippocampus,Shijie Gu,mountainsort4,CA1_tet_Shijie_whiten,eliot20221024_.nwb_07_Seq2Sleep4_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/eliot20221024_.nwb_07_Seq2Sleep4_0_franklab_tetrode_hippocampus_8cafe6a9_spikesorting,1754450919
eliot20221024_.nwb,0,08_Seq2Session5,franklab_tetrode_hippocampus,Shijie Gu,mountainsort4,CA1_tet_Shijie_whiten,eliot20221024_.nwb_08_Seq2Session5_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/eliot20221024_.nwb_08_Seq2Session5_0_franklab_tetrode_hippocampus_be001a09_spikesorting,1745084807
eliot20221024_.nwb,0,09_Seq2Sleep5,franklab_tetrode_hippocampus,Shijie Gu,mountainsort4,CA1_tet_Shijie_whiten,eliot20221024_.nwb_09_Seq2Sleep5_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/eliot20221024_.nwb_09_Seq2Sleep5_0_franklab_tetrode_hippocampus_d36f4ff0_spikesorting,1754452108
eliot20221024_.nwb,0,10_Seq2Session6,franklab_tetrode_hippocampus,Shijie Gu,mountainsort4,CA1_tet_Shijie_whiten,eliot20221024_.nwb_10_Seq2Session6_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/eliot20221024_.nwb_10_Seq2Session6_0_franklab_tetrode_hippocampus_316970f1_spikesorting,1745086109


In [143]:
(SpikeSorting & {"nwb_file_name": nwb_copy_file_name,"sorter":"mountainsort4"}).fetch("sort_interval_name")

array(['02_Rev1Session2', '04_Rev1Session3', '06_Rev1Session4',
       '02_Rev1Session2', '04_Rev1Session3', '06_Rev1Session4',
       '02_Rev1Session2', '04_Rev1Session3', '06_Rev1Session4',
       '02_Rev1Session2', '04_Rev1Session3', '06_Rev1Session4',
       '02_Rev1Session2', '04_Rev1Session3', '06_Rev1Session4',
       '02_Rev1Session2', '04_Rev1Session3', '06_Rev1Session4',
       '02_Rev1Session2', '04_Rev1Session3', '06_Rev1Session4',
       '02_Rev1Session2', '04_Rev1Session3', '06_Rev1Session4',
       '02_Rev1Session2', '04_Rev1Session3', '06_Rev1Session4',
       '02_Rev1Session2', '04_Rev1Session3', '06_Rev1Session4',
       '02_Rev1Session2', '04_Rev1Session3', '06_Rev1Session4',
       '02_Rev1Session2', '04_Rev1Session3', '06_Rev1Session4',
       '02_Rev1Session2', '04_Rev1Session3', '06_Rev1Session4',
       '02_Rev1Session2', '04_Rev1Session3', '06_Rev1Session4',
       '02_Rev1Session2', '04_Rev1Session3', '06_Rev1Session4',
       '02_Rev1Session2', '04_Rev1Sessio